In [16]:
import pathlib
import numpy as np
import scipy as sp
from astropy.io import fits
import matplotlib.pyplot as plt
from scipy import ndimage

In [17]:
here = pathlib.Path('.')
calibration_image = "ubai2505m"
calib_dir = here / "hst-data" / "calibration"

In [18]:
def load_calibration_image(image_id, folder=here):
    """Return {file_type: {'data': array, 'header': primary header}} for one image id.

    Handles the three layouts in the data: a 3D image in the primary HDU, a 3D image
    in an extension, or one 2D image extension per chip (stacked into 3D).
    """
    cal = {}
    for file in sorted(folder.rglob(image_id + "_*.fits")):
        file_type = file.stem.split("_", 1)[1]   # 'ubai2505m_maskfile' -> 'maskfile'
        with fits.open(file) as hdul:
            images = [h.data for h in hdul
                      if isinstance(h, (fits.PrimaryHDU, fits.ImageHDU)) and h.data is not None]
            if not images:
                raise ValueError(f"No image data found in {file}")
            data = images[0] if len(images) == 1 else np.stack(images)
            cal[file_type] = {'data': data.copy(), 'header': hdul[0].header.copy()}
    return cal

In [20]:
cal = load_calibration_image(calibration_image, calib_dir)

for file_type, entry in cal.items():
    print(file_type, entry['data'].shape, entry['data'].dtype)

atodfile (4, 2, 4096) >f4
biasfile (4, 800, 800) >f4
blevfile (4, 800, 14) int16
darkfile (4, 800, 800) >f4
flatfile (4, 800, 800) >f4
maskfile (4, 800, 800) >i2
raw (4, 800, 800) >i2


In [21]:
def apply_mask(cal, defective=2, questionable=256, reject_questionable=False):
    """Mask the raw image with the MASKFILE.

    Defective pixels become NaN. Questionable pixels keep their values unless
    reject_questionable=True, and are returned as a boolean array either way.

    Returns (masked_data, questionable_pixels).
    """
    raw = cal['raw']['data']
    mask = cal['maskfile']['data']

    if raw.shape != mask.shape:
        raise ValueError(f"Raw shape {raw.shape} does not match mask shape {mask.shape}")

    unexpected = set(np.unique(mask)) - {0, defective, questionable}
    if unexpected:
        raise ValueError(f"Mask contains unexpected values: {sorted(unexpected)}")

    data = raw.astype(np.float64)   # new array: the raw data stays untouched, and floats can hold NaN

    bad = mask == defective
    is_questionable = mask == questionable
    if reject_questionable:
        bad = bad | is_questionable

    data[bad] = np.nan
    return data, is_questionable


def save_fits(data, header, path, history=None):
    """Save data with a copy of the header, adding a HISTORY line describing the step."""
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    header = header.copy()
    if history:
        header['HISTORY'] = history
    fits.writeto(path, data, header, overwrite=True)

In [22]:
masked, questionable = apply_mask(cal)

# Check: every defective pixel should now be NaN, and nothing else
print("defective in mask:", np.sum(cal['maskfile']['data'] == 2))
print("NaN in result:    ", np.sum(np.isnan(masked)))
print("questionable kept:", np.sum(questionable))

defective in mask: 5430
NaN in result:     5430
questionable kept: 13378


In [23]:
save_fits(masked, cal['raw']['header'],
          here / "hst-processed" / f"{calibration_image}_masked.fits",
          history="MASKFILE applied: defective (2) set to NaN, questionable (256) kept")